# 04 — Manual Labeling

**Day 4 — May 22, 2026**

Labels 500 translated Olist reviews across three schemas:
- Sentiment: positive / negative / neutral
- Theme: delivery / quality / service / price / returns / other
- Journey stage: pre-purchase / purchase / delivery / post-purchase

Uses ipywidgets — no Doccano, no separate server, works directly in Jupyter.
Labels save to disk after every single review so nothing is lost if the
kernel crashes.

In [5]:
import json
import sys
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# path fix — works whether you're in notebooks/ or project root
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.append(str(ROOT))

PROCESSED  = ROOT / "data" / "processed"
LABELED    = ROOT / "data" / "labeled"
LABELED.mkdir(exist_ok=True)

SAMPLE_OUT  = LABELED / "labeling_sample.jsonl"
LABELS_OUT  = LABELED / "labeled_500.parquet"

## Step 1 — Build stratified sample (500 reviews)
Proportional across star ratings so all sentiment classes are represented.
Run once. If labeling_sample.jsonl already exists, skip to Step 2.

In [6]:
df = pd.read_parquet(PROCESSED / "olist_reviews_translated.parquet")

sample = (
    df.groupby("review_score", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 100), random_state=42))
    .head(500)
    .reset_index(drop=True)
)

print("Sample size:", len(sample))
print("Score distribution:")
print(sample["review_score"].value_counts().sort_index())

# save as JSONL (portfolio artifact — shows you did the prep work)
with open(SAMPLE_OUT, "w", encoding="utf-8") as f:
    for _, r in sample.iterrows():
        f.write(json.dumps({
            "text": r["review_clean"],
            "meta": {"order_id": r["order_id"], "score": int(r["review_score"])}
        }) + "\n")
print(f"\nSaved sample to {SAMPLE_OUT}")

Sample size: 500
Score distribution:
review_score
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64

Saved sample to C:\Users\akskumari\Desktop\cx-analytics-project\data\labeled\labeling_sample.jsonl


C:\Users\akskumari\AppData\Local\Temp\ipykernel_21052\10563916.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 100), random_state=42))


## Step 2 — Load existing labels (safe to re-run)
If you've already labeled some reviews and are resuming, this loads them.
First run: starts fresh.

In [7]:
if LABELS_OUT.exists():
    labels_df = pd.read_parquet(LABELS_OUT)
    labeled_ids = set(labels_df["order_id"].tolist())
    print(f"Resuming — {len(labeled_ids)} already labeled, "
          f"{len(sample) - len(labeled_ids)} remaining")
else:
    labels_df = pd.DataFrame(columns=[
        "order_id", "review_score", "review_clean",
        "sentiment", "theme", "journey_stage"
    ])
    labeled_ids = set()
    print("Starting fresh — 0 labels so far")

Resuming — 300 already labeled, 200 remaining


## Step 3 — Labeling UI

**How to use:**
- Read the review text carefully
- Click one button per schema (sentiment, theme, journey stage)
- Hit **Save & Next** — labels write to disk immediately
- Hit **Skip** if the review is too short or ambiguous to label confidently
- The progress bar shows how far you are

**Quick reference:**
- Sentiment: does the customer feel positive, negative, or neutral?
- Theme: what is the review *primarily* about? Pick the dominant one.
- Journey stage: where in the purchase journey is this review about?

Run this cell to start (or resume) labeling.

In [9]:
todo = sample[~sample["order_id"].isin(labeled_ids)].reset_index(drop=True)
print(f"Reviews to label: {len(todo)}")

if len(todo) == 0:
    print("All done! Jump to Step 4 to check your distribution.")
else:
    # --- state ---
    state = {"idx": 0, "sentiment": None, "theme": None, "journey": None}

    # --- layout helpers ---
    def make_toggle(options, desc):
        return widgets.ToggleButtons(
            options=options,
            description=desc,
            button_style="",
            style={"description_width": "120px",
                   "button_width": "110px"}
        )

    # --- widgets ---
    progress    = widgets.IntProgress(
                    value=0, min=0, max=len(todo),
                    description="Progress:",
                    style={"description_width": "70px"},
                    layout=widgets.Layout(width="60%"))
    progress_lbl = widgets.Label(value=f"0 / {len(todo)}")

    score_lbl   = widgets.Label(value="")
    review_box  = widgets.Textarea(
                    value="", disabled=True,
                    layout=widgets.Layout(width="100%", height="120px"))

    sentiment_w = make_toggle(
                    ["positive", "negative", "neutral"], "Sentiment:")
    theme_w     = make_toggle(
                    ["delivery", "quality", "service",
                     "price", "returns", "other"], "Theme:")
    journey_w   = make_toggle(
                    ["pre-purchase", "purchase",
                     "delivery", "post-purchase"], "Journey stage:")

    save_btn    = widgets.Button(
                    description="Save & Next",
                    button_style="success",
                    layout=widgets.Layout(width="150px"))
    skip_btn    = widgets.Button(
                    description="Skip",
                    button_style="warning",
                    layout=widgets.Layout(width="100px"))
    status_out  = widgets.Output()

    def load_review(idx):
        if idx >= len(todo):
            with status_out:
                clear_output()
                print("All reviews labeled! Jump to Step 4.")
            return
        r = todo.iloc[idx]
        score_lbl.value  = (f"Review {idx + 1} / {len(todo)}   "
                            f"⭐ {int(r['review_score'])} star")
        review_box.value = str(r["review_clean"])
        # reset toggles to first option
        sentiment_w.value = "positive"
        theme_w.value     = "delivery"
        journey_w.value   = "pre-purchase"

    def save_label(row, sentiment, theme, journey):
        global labels_df
        new_row = pd.DataFrame([{
            "order_id":     row["order_id"],
            "review_score": row["review_score"],
            "review_clean": row["review_clean"],
            "sentiment":    sentiment,
            "theme":        theme,
            "journey_stage": journey
        }])
        labels_df = pd.concat([labels_df, new_row], ignore_index=True)
        labels_df.to_parquet(LABELS_OUT, index=False)

    def on_save(b):
        idx = state["idx"]
        if idx >= len(todo):
            return
        row = todo.iloc[idx]
        save_label(row, sentiment_w.value, theme_w.value, journey_w.value)
        state["idx"] += 1
        progress.value    = state["idx"]
        progress_lbl.value = f"{state['idx']} / {len(todo)}"
        with status_out:
            clear_output()
            print(f"Saved: {sentiment_w.value} | {theme_w.value} "
                  f"| {journey_w.value}")
        load_review(state["idx"])

    def on_skip(b):
        state["idx"] += 1
        progress.value    = state["idx"]
        progress_lbl.value = f"{state['idx']} / {len(todo)}"
        with status_out:
            clear_output()
            print("Skipped.")
        load_review(state["idx"])

    save_btn.on_click(on_save)
    skip_btn.on_click(on_skip)

    load_review(0)

    display(widgets.VBox([
        widgets.HBox([progress, progress_lbl]),
        score_lbl,
        review_box,
        sentiment_w,
        theme_w,
        journey_w,
        widgets.HBox([save_btn, skip_btn]),
        status_out
    ]))

Reviews to label: 200


## Step 4 — Check label distribution
Run after you finish (or mid-session to check your calibration).
No class should be under 30 samples — if one is, you may need more reviews.

In [10]:
if LABELS_OUT.exists():
    done = pd.read_parquet(LABELS_OUT)
    print(f"Total labeled: {len(done)}\n")

    print("Sentiment distribution:")
    print(done["sentiment"].value_counts())
    print()
    print("Theme distribution:")
    print(done["theme"].value_counts())
    print()
    print("Journey stage distribution:")
    print(done["journey_stage"].value_counts())
    print()

    # flag thin classes
    for col in ("sentiment", "theme", "journey_stage"):
        thin = done[col].value_counts()
        thin = thin[thin < 30]
        if not thin.empty:
            print(f"WARNING — thin classes in {col} (under 30 samples):")
            print(thin)
else:
    print("No labels saved yet — run Step 3 first.")

Total labeled: 500

Sentiment distribution:
sentiment
negative    271
positive    176
neutral      53
Name: count, dtype: int64

Theme distribution:
theme
delivery    236
quality     159
other        51
service      37
returns      12
price         5
Name: count, dtype: int64

Journey stage distribution:
journey_stage
post-purchase    283
delivery         208
pre-purchase       9
Name: count, dtype: int64

WARNING — thin classes in theme (under 30 samples):
theme
returns    12
price       5
Name: count, dtype: int64
WARNING — thin classes in journey_stage (under 30 samples):
journey_stage
pre-purchase    9
Name: count, dtype: int64


## Step 5 — Export to JSONL (portfolio artifact)
Saves a JSONL version of labeled data matching what Doccano would have
produced — so your downstream code and resume narrative stay consistent.

In [11]:
if LABELS_OUT.exists():
    done = pd.read_parquet(LABELS_OUT)
    jsonl_out = LABELED / "labeled_500.jsonl"
    with open(jsonl_out, "w", encoding="utf-8") as f:
        for _, r in done.iterrows():
            f.write(json.dumps({
                "text": r["review_clean"],
                "label": {
                    "sentiment":     r["sentiment"],
                    "theme":         r["theme"],
                    "journey_stage": r["journey_stage"]
                },
                "meta": {
                    "order_id":     r["order_id"],
                    "review_score": int(r["review_score"])
                }
            }) + "\n")
    print(f"Exported {len(done)} labeled reviews to {jsonl_out}")
else:
    print("No labels yet — finish Step 3 first.")

Exported 500 labeled reviews to C:\Users\akskumari\Desktop\cx-analytics-project\data\labeled\labeled_500.jsonl
